In [1]:
import pandas as pd
import numpy as np
from preprocess import preprocess
from race_state import RaceState
from simulation import simulate_one_scenario

In [2]:
import pickle

with open('../Models.ipynb/models/pace_predictor_rf_final_partie2.pkl', 'rb') as f:
    rf = pickle.load(f)

df_processed = pd.read_csv('master_dataset_partie2_2024_stint_preprocessed.csv')

print('Modele RF charge dans la variable rf')
print(f'df_processed charge: {df_processed.shape[0]} lignes, {df_processed.shape[1]} colonnes')

Modele RF charge dans la variable rf
df_processed charge: 17799 lignes, 21 colonnes


In [3]:
import matplotlib.pyplot as plt

def validate_monte_carlo(state, model, df_processed, iterations=200):
    gaps = []
    for _ in range(iterations):
        # On simule la stratégie réelle pour valider
        gap = simulate_one_scenario(state, model, 'STAY_OUT', 0, df_processed, noise_level=0.1)
        gaps.append(gap)
    
    plt.hist(gaps, bins=20, edgecolor='black')
    plt.axvline(np.mean(gaps), color='red', linestyle='dashed', label=f'Moyenne: {np.mean(gaps):.2f}s')
    plt.title("Distribution des Gaps Finaux (Validation)")
    plt.xlabel("Gap vs Rival (secondes)")
    plt.ylabel("Fréquence")
    plt.legend()
    plt.show()

# Si la cloche est centrée sur le vrai résultat de la course, ton simulateur est validé !

In [ ]:
# --- Construire un state initial pour validate_monte_carlo ---
race_number = 38   # exemple: Italian GP 2024
total_laps = 57
target_lap = 20
initial_gap = 3.2

race_df = df_processed[df_processed['RaceNumber'] == race_number].copy()
if race_df.empty:
    raise ValueError(f'Aucune ligne trouvée pour RaceNumber={race_number}')

# On prend un tour valide qui contient au moins 2 voitures
lap_counts = race_df.groupby('LapNumber').size()
eligible_laps = lap_counts[lap_counts >= 2].index.tolist()
if not eligible_laps:
    raise ValueError('Impossible de créer un state: aucun LapNumber avec au moins 2 lignes.')

if target_lap not in eligible_laps:
    target_lap = int(sorted(eligible_laps)[0])

lap_df = race_df[race_df['LapNumber'] == target_lap].copy().reset_index(drop=True)
driver_row = lap_df.iloc[0]
rival_row = lap_df.iloc[1]

state = RaceState.from_dataset_row(
    driver_row=driver_row,
    rival_row=rival_row,
    total_laps=total_laps,
    gap=initial_gap,
)

print(f'state cree: race={state.race_number}, lap={state.lap}/{state.total_laps}, gap={state.gap_to_rival:.2f}s')
print(f'compound joueur={state.compound} (C{state.compound_hardness}), tyre_life={state.tyre_life}')
print(f'compound rival hardness=C{state.rival_compound_hardness}, rival_tyre_life={state.rival_tyre_life}')

# Utilisation:
# validate_monte_carlo(state, rf, df_processed, iterations=200)